- 실습 기본 환경 설정


In [1]:

# 예제 실행 및 시각화를 위한 공통 라이브러리 로딩

# 코랩 환경 등 깃허브 전체를 clone해서 실습하는 경우가 아니라면 
# 공통 라이브러리를 불러오기 위해서 별도의 과정이 필요하므로 code_reference/README.md 파일을 확인하자.
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

# 시각화 결과를 파일에 저장하지 않음
viz.configure(save_grayscale=False)

# matplotlib 시각화에서 한글 폰트 사용 설정
common.set_korean_plot_env()

# 재현성 보장을 위한 시드 고정
#   여기서 재현성은 '동일 컴퓨터, 동일 버전의 파이썬, 동일 버전의 파이토치' 환경에서 재현이 가능하다는 의미로, 
#   독자의 결과는 저자의 결과와 달라질 수 있다는 점을 밝혀둔다.
#   또한 같은 환경에서도 GPU를 사용하는 경우, 일부 연산의 비결정적 성질로 인해 실행시마다 결과가 조금씩 달라질 수 있다.
SEED = 42
common.set_seed(SEED)

# 실습 환경에 맞는 하드웨어 가속기 장치 객체
device = common.get_device()

CUDA를 사용합니다.


# 12-4 검색 증강 생성으로 환각 줄이기

본 노트북은 12-4절 본문 예제 [코드 12-18]~[코드 12-22]를 모아 위에서 아래로
실행한다. 문장 임베딩 모델로 문서를 인덱싱하고, 코사인 유사도로 관련 문서를
검색해 LLM에 함께 전달하는 간단한 RAG 시스템(모델 18: 진실된 인공지능 대화
서비스)을 만든다.

**API 키 안내** - 생성 단계는 기본적으로 Groq API를 사용한다. Groq 콘솔
(https://console.groq.com)에서 무료 API 키를 발급해 `GROQ_API_KEY`에 넣으면
실제 API를 호출한다. 키가 없으면 같은 인터페이스를 그대로 유지한 채 로컬 한국어
LLM(Bllossom-3B)으로 생성 단계를 대체하므로, 키 없이도(GPU 환경) 실제 RAG
결과를 확인할 수 있다. 다만 로컬 3B 모델은 Groq의 8B 모델보다 품질이 낮아 본문
[표 12-12]의 결과와 답변이 다를 수 있다.

**이 장의 코드 컨벤션 알림** - 12장은 허깅페이스 라이브러리를 중심 주제로
다루므로, `MODEL_NAME`, `GROQ_MODEL` 등 상수 표기는 공통 컨벤션을 따르지만
`model`, `embed_model`, `groq_client` 같은 객체명은 통용 표기를 그대로 사용한다.

다루는 내용
- 본문 비공개 항목: 예제용 근거 문서 6편(documents), Groq 키가 없을 때의 로컬 LLM 폴백
- [코드 12-18] 문서 임베딩 생성
- [코드 12-19] 질문과 가장 유사한 문서 검색
- [코드 12-20] Groq 클라이언트 생성과 호출
- [코드 12-21] RAG 프롬프트 구성과 생성 함수
- [코드 12-22] 통합 RAG 파이프라인 + RAG/비-RAG 답변 비교 ([표 12-12] 대응)

> 사용 모델: snunlp/KR-SBERT-V40K-klueNLI-augSTS (MIT License),
> Groq llama-3.1-8b-instant (Llama 3.1 커뮤니티 라이선스),
> 로컬 폴백 Bllossom/llama-3.2-Korean-Bllossom-3B (Llama 3.2 커뮤니티 라이선스).

이번 절은 본문의 모델 16의 구현을 포함한다. 모델 16의 제시문은 다음과 같다.

> **모델 16. 진실한 인공지능 대화 서비스**
>
> LLM에 외부 문서를 연결해 답변의 근거를 보강하는 간단한 RAG 시스템을 만들어 본다. 답변 근거 문서로 LLM과 트랜스포머에 관한 여섯 개의 짧은 문자열을 딕셔너리 형태로 제공한다. RAG 답변을 생성하는 LLM은 직접 불러오는 대신 외부 API로 제공되는 LLM 서비스를 활용한다.


In [2]:
# 참고 - Groq API 키 설정
# 발급받은 키가 있으면 아래 변수에 직접 넣거나, 셸에서 환경 변수로 설정한다.
#   리눅스/macOS:  export GROQ_API_KEY='발급받은 키'
#   주피터 노트북: os.environ['GROQ_API_KEY'] = '발급받은 키'
# 키가 비어 있으면 이 노트북은 로컬 한국어 LLM(Bllossom-3B)으로 생성 단계를 대체한다.
import os

GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')  # 'groq_key' 자리에 키 입력
USE_LOCAL_LLM = not bool(GROQ_API_KEY)
print(f'로컬 LLM 폴백 모드: {USE_LOCAL_LLM}')

로컬 LLM 폴백 모드: True


## 참고 - 예제용 근거 문서 집합 정의

답변 근거가 될 LLM·트랜스포머 관련 짧은 글 6편을 딕셔너리(`documents`)로 둔다.
실제 서비스라면 수집·청킹 단계를 거치지만, 예제는 이를 생략하고 짧은 문서를
직접 코드 안에 둔다.


In [3]:
# 참고 - LLM·트랜스포머 관련 짧은 글 6편
documents = [
    {'title': '트랜스포머 아키텍처',
     'content': (
         '트랜스포머는 2017년 구글이 발표한 신경망 구조로, 어텐션 메커니즘만으로 '
         '순차 데이터를 처리한다. 인코더와 디코더가 모두 셀프 어텐션과 피드포워드 '
         '계층의 반복으로 구성되며, 위치 정보는 위치 인코딩을 통해 더한다. 이후 '
         '거의 모든 대규모 언어 모델의 기본 구조가 되었다.')},
    {'title': 'GPT 시리즈',
     'content': (
         'GPT는 OpenAI가 공개한 디코더 전용 트랜스포머 계열의 언어 모델 시리즈로, '
         'GPT-3(2020)에서 1,750억 파라미터로 규모를 크게 키워 화제가 됐다. GPT-4는 '
         '멀티모달 입력을 지원하며 ChatGPT 서비스의 기반 모델로 사용된다. 대규모 '
         '사전 학습 후 지시어 미세 조정과 RLHF로 정렬되는 파이프라인을 따른다.')},
    {'title': 'LLaMA',
     'content': (
         'LLaMA는 Meta가 2023년부터 공개한 오픈 가중치 대규모 언어 모델 시리즈다. '
         'LLaMA 2와 LLaMA 3로 이어지며 연구·상용 모두 사용 가능한 라이선스로 배포돼 '
         '오픈소스 LLM 생태계의 표준이 됐다. 한국어 특화 파생 모델인 Bllossom도 '
         'LLaMA 3 계열을 기반으로 한다.')},
    {'title': 'RAG(검색 증강 생성)',
     'content': (
         'RAG는 2020년 메타(Meta) AI 연구팀이 발표한 기법으로, LLM의 환각과 학습 후 '
         '정보 부재 문제를 외부 문서 검색으로 보완한다. 파이프라인은 (1) 문서를 임베딩 '
         '벡터로 변환해 저장, (2) 질문을 임베딩해 유사한 문서를 검색, (3) 검색된 문서를 '
         '컨텍스트로 LLM에 전달해 답변을 생성하는 세 단계로 구성된다.')},
    {'title': 'Groq',
     'content': (
         'Groq는 LPU(Language Processing Unit)라는 AI 추론 전용 칩을 개발한 '
         '스타트업이다. 오픈소스 LLM의 API 서비스를 함께 제공하며, 호출 인터페이스가 '
         'OpenAI API와 동일해 코드 호환성이 높다. 무료 티어에서도 학습·실험용으로 '
         '충분한 사용량을 제공해 RAG 같은 빠른 응답이 필요한 시스템에 적합하다.')},
    {'title': '파인튜닝과 프롬프트 엔지니어링',
     'content': (
         '파인튜닝은 사전 학습 모델을 작업 데이터로 추가 학습해 모델의 동작 자체를 '
         '바꾸는 방법이다. 반면 프롬프트 엔지니어링은 모델은 그대로 두고 입력 프롬프트를 '
         '잘 구성해 원하는 출력을 끌어내는 방법이다. RAG는 후자의 발전된 형태로, 모델은 '
         '그대로 두고 외부 문서를 동적으로 컨텍스트에 끼워 넣어 답변을 보강한다.')},
]
print(f'문서 개수: {len(documents)}')
for doc in documents:
    print(f"- {doc['title']} ({len(doc['content'])}자)")


문서 개수: 6
- 트랜스포머 아키텍처 (152자)
- GPT 시리즈 (183자)
- LLaMA (163자)
- RAG(검색 증강 생성) (182자)
- Groq (187자)
- 파인튜닝과 프롬프트 엔지니어링 (174자)


## [코드 12-18] 문서 임베딩 생성

한국어 문장 유사도에 특화된 KR-SBERT-KLUE 모델을 `SentenceTransformer`로
불러와 문서 임베딩을 만든다. 코사인 유사도를 내적으로 계산하기 위해 L2 정규화해
둔다.


In [4]:
###############################################################################
# 코드 12-18 - 문서 임베딩 생성
###############################################################################

import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')


def build_index(documents, model):
    texts = [doc['content'] for doc in documents]
    embeddings = model.encode(texts, convert_to_tensor=True)
    # L2 정규화: 코사인 유사도를 내적으로 계산하기 위한 전처리
    embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings


doc_embeddings = build_index(documents, embed_model)
print(f'임베딩 shape: {doc_embeddings.shape}')   # (문서 수, 임베딩 차원)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

임베딩 shape: torch.Size([6, 768])


## [코드 12-19] 질문과 가장 유사한 문서 검색

질문도 같은 방식으로 임베딩·정규화한 뒤, 정규화된 문서 임베딩과의 행렬곱 한
번으로 모든 문서의 코사인 유사도를 구한다. `topk`로 상위 문서를 고른다.


In [5]:
# 코드 12-19 - 질문과 가장 유사한 문서 검색
import torch

def retrieve(query, documents, doc_embeddings, embed_model, top_k=2):
    # doc_embeddings는 build_index()에서 이미 L2 정규화된 상태로 전달받는다고 가정.
    # 질문도 같은 방식으로 임베딩하고 정규화한다.
    query_embedding = embed_model.encode(query, convert_to_tensor=True)
    # 1차원 벡터이므로 dim=0 (배치 텐서면 dim=1 또는 차원과 무관하게 dim=-1)
    query_embedding = F.normalize(query_embedding, p=2, dim=0)
    # 행렬 곱으로 코사인 유사도 계산 (정규화된 벡터의 내적 = 코사인 유사도)
    scores = torch.matmul(doc_embeddings, query_embedding)
    top_indices = torch.topk(scores, k=top_k).indices.tolist()
    return ([documents[i] for i in top_indices],
            [scores[i].item() for i in top_indices])


retrieved_docs, scores = retrieve(
    '트랜스포머 모델은 어떤 구조인가요?', documents, doc_embeddings, embed_model,
)
for doc, score in zip(retrieved_docs, scores):
    print(f'[유사도: {score:.4f}] {doc["title"]}')


[유사도: 0.3957] 트랜스포머 아키텍처
[유사도: 0.3523] GPT 시리즈


## 참고 - 생성 클라이언트 정의 (Groq 또는 로컬 LLM)

`GROQ_API_KEY`가 있으면 실제 `groq.Groq` 클라이언트를 사용한다. 키가 없으면 로컬
한국어 LLM(Bllossom-3B)을 불러와 `chat.completions.create()`와 동일한 인터페이스를
제공하는 드롭인 클라이언트를 만든다. 덕분에 이후 [코드 12-20]~[코드 12-22]는 생성
백엔드와 무관하게 그대로 동작한다(코드 12-22의 `generate_fn` 추상화와 같은 맥락).

In [6]:
# 참고 - 생성 클라이언트 (실제 Groq API 또는 로컬 LLM 폴백)


GROQ_MODEL = 'llama-3.1-8b-instant'
MAX_GEN_TOKEN = 512

if not USE_LOCAL_LLM:
    from groq import Groq
    groq_client = Groq(api_key=GROQ_API_KEY)
else:
    # Groq API 키가 없을 때: 로컬 한국어 LLM(Bllossom-3B)으로 생성 단계를 대체한다.
    # groq_client.chat.completions.create(...)와 동일한 인터페이스를 제공하는
    # 드롭인 클라이언트라, 이후 코드는 백엔드와 무관하게 그대로 동작한다.
    from types import SimpleNamespace

    from transformers import AutoModelForCausalLM, AutoTokenizer

    LOCAL_MODEL_NAME = 'Bllossom/llama-3.2-Korean-Bllossom-3B'
    print(f'Groq API 키가 없어 로컬 LLM({LOCAL_MODEL_NAME})으로 생성합니다. '
          '최초 1회 모델 다운로드와 로딩에 시간이 걸릴 수 있습니다.')

    _local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)
    _local_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_NAME, torch_dtype=torch.float16,
    ).to(device)
    _local_model.eval()
    # VRAM이 빠듯하면(8GB 미만) [코드 12-14]의 BitsAndBytesConfig로 4비트 로드도 가능

    # Llama 3 계열 종료 토큰: 문장 끝 토큰과 <|eot_id|>를 함께 종료 신호로 사용
    _eot_id = _local_tokenizer.convert_tokens_to_ids('<|eot_id|>')
    _terminators = list({t for t in (_local_tokenizer.eos_token_id, _eot_id)
                         if isinstance(t, int) and t >= 0})

    class _LocalChat:
        @staticmethod
        def create(model, messages, max_tokens=MAX_GEN_TOKEN,
                   temperature=0.1, **kw):
            # 대화틀(chat template)로 messages를 모델 입력 형식으로 인코딩
            inputs = _local_tokenizer.apply_chat_template(
                messages, add_generation_prompt=True,
                return_tensors='pt', return_dict=True,
            ).to(device)
            with torch.no_grad():
                output_ids = _local_model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    do_sample=False,            # 일관된 결과를 위해 그리디 디코딩
                    eos_token_id=_terminators,
                    pad_token_id=_local_tokenizer.eos_token_id,
                )
            # 새로 생성된 토큰만 디코딩(입력 프롬프트 부분 제외)
            input_len = inputs['input_ids'].shape[-1]
            generated = output_ids[0][input_len:]
            text = _local_tokenizer.decode(generated, skip_special_tokens=True)
            return SimpleNamespace(
                choices=[SimpleNamespace(
                    message=SimpleNamespace(content=text.strip()))])

    class _LocalClient:
        def __init__(self):
            self.chat = SimpleNamespace(completions=_LocalChat())

    groq_client = _LocalClient()

Groq API 키가 없어 로컬 LLM(Bllossom/llama-3.2-Korean-Bllossom-3B)으로 생성합니다. 최초 1회 모델 다운로드와 로딩에 시간이 걸릴 수 있습니다.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

## [코드 12-20] Groq 클라이언트 생성과 호출

`chat.completions.create()`는 JSON 응답을 파이썬 객체(`ChatCompletion`)로 변환해
반환하며, `choices[0].message.content`에서 본문을 꺼낸다. 본문 [코드 12-20]은
키를 직접 지정하지만(가독성 우선), 노트북은 환경 변수로 키를 읽거나 키가 없으면
로컬 LLM으로 안전하게 대체한다.

In [7]:
# 코드 12-20 - Groq 클라이언트 생성과 호출
response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{'role': 'user', 'content': '트랜스포머 모델은 어떤 구조인가요?'}],
    max_tokens=MAX_GEN_TOKEN,
    temperature=0.1,                 # 일관성을 위해 낮은 온도
)
print('생성 결과:\n' + response.choices[0].message.content)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


생성 결과:
트랜스포머 모델은 주로 자연어 처리(NLP) 분야에서 사용되는 deep learning 모델입니다. 이 모델은 주로 '트랜스포머(Transformer)'이라는 이름의 구조를 기반으로 합니다. 트랜스포머 모델은 주로 'Self-Attention Mechanism'을 사용하여 문장을 처리합니다.

트랜스포머 모델의 주요 구조는 다음과 같습니다:

1. **Input Embedding Layer**: 이 단계에서는 입력 텍스트를 숫자로 변환하여 모델이 처리할 수 있도록 합니다. 이 단계에서는 텍스트의 단어를 벡터로 변환하여 입력 데이터를 준비합니다.

2. **Self-Attention Mechanism**: 이 단계에서는 모델이 입력 데이터의 각 단어 간의 관계를 파악합니다. Self-Attention Mechanism은 각 단어가 다른 단어와의 관계를 강조하여 문장의 의미를 이해하는 데 도움을 줍니다.

3. **Encoder Layer**: 이 단계에서는 Self-Attention Mechanism을 사용하여 입력 데이터를 처리합니다. 각 단어의 벡터를 처리하고, 각 단어 간의 관계를 파악하여 문장의 의미를 이해합니다.

4. **Decoder Layer**: 이 단계에서는 모델이 출력을 생성하는 데 사용됩니다. Decoder Layer는 Self-Attention Mechanism을 사용하여 입력 데이터를 기반으로 출력을 생성합니다.

5. **Output Layer**: 이 단계에서는 모델이 생성한 출력을 변환하여 원래의 텍스트 형태로 변환합니다.

트랜스포머 모델은 주로 'Bert', 'RoBERTa', 'DistilBERT' 등 다양한 버전으로 개발되었습니다. 각 버전은 특정한 목적을 위해 설계되었습니다. 예를 들어, 'Bert'는 기본적인 NLP 작업을 수행하는 데 사용되며, 'RoBERTa'는 더 나은 성능을 제공하는 버전입니다.

트랜스포머 모델은 매우 강력한 자연어 처리 능력을 가지고 있으며, 다양한 NLP 작업에서 사용됩니다. 예를

## [코드 12-21] RAG 프롬프트 구성과 생성 함수

시스템 메시지에는 어조·안전 가이드라인을, 사용자 메시지에는 검색된 문서와 그
문서를 참고해 답하라는 지시를 담는다. 맨 앞의 "문서에 없는 내용은 추측하지 말라"는
한 줄이 RAG의 핵심 제어 장치다.


In [8]:
# 코드 12-21 - RAG 프롬프트 구성과 생성 함수
def generate_with_groq(prompt, model=GROQ_MODEL, max_tokens=MAX_GEN_TOKEN):
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system',
             'content': '전문성 있는 한국어 문장으로 답변하며, 자료가 제공되지 '
                        '않은 사항은 결코 추측해 답하지 말고 모른다고 응답한다.'},
            {'role': 'user', 'content': prompt},
        ],
        max_tokens=max_tokens,
        temperature=0.1,
    )
    return response.choices[0].message.content


def build_rag_prompt(query, retrieved_docs):
    # 검색된 문서를 컨텍스트로 결합
    context = '\n\n'.join([
        f"[문서 {i + 1}: {doc['title']}]\n{doc['content']}"
        for i, doc in enumerate(retrieved_docs)
    ])
    # 컨텍스트와 질문을 한 프롬프트로 묶고, 답변 지침을 맨 앞에 둔다.
    prompt = (
        '다음 문서에 없는 내용은 결코 추측해 답하지 말고 모른다고 답하며, '
        '문서를 참고해 질문에 답한다.\n\n'
        f'* 참고 문서\n{context}\n\n'
        f'* 질문\n{query}'
    )
    return prompt


## [코드 12-22] 통합 RAG 파이프라인

검색 파이프라인과 생성 파이프라인을 한 함수로 묶는다. `generate_fn`을 인자로
받아 생성 함수만 교체할 수 있게 한다.


In [9]:
# 코드 12-22 - 통합 RAG 파이프라인
def rag_pipeline(query, documents, doc_embeddings, embed_model,
                 generate_fn, top_k=2):
    # 1단계: 검색
    retrieved_docs, scores = retrieve(
        query, documents, doc_embeddings, embed_model, top_k=top_k,
    )
    # 2단계: 프롬프트 구성
    prompt = build_rag_prompt(query, retrieved_docs)
    # 3단계: 생성
    answer = generate_fn(prompt)
    return answer, retrieved_docs, scores


### 참고 - RAG 적용/미적용 답변 비교 ([표 12-12] 대응)

성격이 다른 세 질문((1) 최신 회사 정보, (2) 환각 유도, (3) 문서 범위 밖)으로
RAG 적용/미적용 답변을 비교한다. Groq(8B)를 사용하면 본문 [표 12-12]에 가까운
결과가 나온다. 로컬 폴백(Bllossom-3B)은 모델이 작아 문서 범위 밖 질문을 또렷이
거절하지 못하는 등 답변이 다를 수 있다.

In [10]:
# 참고 - RAG vs 비-RAG 답변 비교 ([표 12-12] 대응)
test_queries = [
    'Groq는 어떤 회사이고, 어떤 모델을 API로 제공하나요?',       # 최신 정보
    'RAG는 누가 언제 발표했나요? 파이프라인은 어떻게 구성되나요?',  # 환각 유도
    '파이토치와 텐서플로의 차이점은 무엇인가요?',                # 문서 범위 밖
]

for q in test_queries:
    print('=' * 60)
    print(f'질문: {q}\n')
    print('--- RAG 없이 (LLM 단독) ---')
    print(generate_with_groq(q))
    print('\n--- RAG 적용 ---')
    answer, docs, rag_scores = rag_pipeline(
        q, documents, doc_embeddings, embed_model, generate_with_groq,
    )
    titles = [d['title'] for d in docs]
    print(f'(검색된 문서: {titles}, '
          f'유사도: {[round(s, 3) for s in rag_scores]})')
    print(answer, '\n')


질문: Groq는 어떤 회사이고, 어떤 모델을 API로 제공하나요?

--- RAG 없이 (LLM 단독) ---
Groq는 AI 기반의 데이터 모델링 플랫폼입니다. Groq는 데이터 모델링을 쉽고 빠르게 할 수 있도록 도와주는 기술을 제공합니다. Groq는 API를 통해 데이터 모델을 생성하고, 이를 통해 데이터베이스와 같은 데이터 스토어와 연동할 수 있습니다. Groq는 데이터 모델링을 자동화하는 데 도움을 주며, 이를 통해 데이터베이스 설계, 데이터 모델링, 데이터 전달 등 다양한 데이터 관리 작업을 효율적으로 수행할 수 있습니다.

--- RAG 적용 ---
(검색된 문서: ['Groq', 'GPT 시리즈'], 유사도: [0.477, 0.46])
Groq는 LPU(Language Processing Unit)라는 AI 추론 전용 칩을 개발한 스타트업입니다. Groq는 오픈소스 LLM의 API 서비스를 제공하며, 호출 인터페이스가 OpenAI API와 동일해 코드 호환성이 높습니다. 

질문: RAG는 누가 언제 발표했나요? 파이프라인은 어떻게 구성되나요?

--- RAG 없이 (LLM 단독) ---
RAG(Reactive Application Gateway)는 Microsoft에서 제공하는 웹 애플리케이션 게이트웨이입니다. RAG는 2019년에 발표되었습니다. 

RAG는 파이프라인을 통해 여러 웹 애플리케이션을 관리하고, 트래픽을 분산하여 성능을 향상시키는 데 사용됩니다. RAG의 파이프라인 구성은 다음과 같습니다:

1. **트래픽 수집**: RAG는 HTTP 트래픽을 수집하여 각 애플리케이션에 전달합니다.
2. **트래픽 분산**: RAG는 트래픽을 여러 애플리케이션에 분산하여 성능을 향상시킵니다.
3. **응답 처리**: RAG는 각 애플리케이션의 응답을 수집하고, 응답을 전달합니다.
4. **데이터 분석**: RAG는 트래픽 데이터를 분석하여 성능을 모니터링하고, 최적화할 수 있는 정보를 제공합니다.

RAG는 Azure와 같은 클라우드 플랫폼에